### 1. Tokenizing text:
#### convert raw text into model-ready training examples
### 2. input-target pairs:
#### build input-target pairs for next-token predictions
### 3. Batching:
#### use fixed shapes to work efficiently with JAX and jit
### 4. Build a Data iterator
### 5. Verify batch shapes and outputs

In [1]:
import jax
import jax.numpy as jnp

import grain.python as grain

import tiktoken
from pathlib import Path
from helper import load_stories_from_file

In [2]:
file_path = Path("validation.csv")
stories = load_stories_from_file(file_path, max_stories=1000)

print("First story (300 chars):\n")
story = stories[0]
print(story[:300], "...")

print(f"\nLoaded stories: {len(stories):,}")


First story (300 chars):

Spot. Spot saw the shiny car and said, "Wow, Kitty, your car is so bright and clean!" Kitty smiled and replied, "Thank you, Spot. I polish it every day."

After playing with the car, Kitty and Spot felt thirsty. They found a small pond with clear water. They drank the water and felt very happy. They ...

Loaded stories: 1,000


In [3]:
tokenizer = tiktoken.get_encoding("gpt2")
print(f"Vocab Size:: {tokenizer.n_vocab:,}")
print(f"Special tokens: {tokenizer.special_tokens_set}")
#tokenizer.n_vocab : How many different token IDs exist?
# special tokens: e.g. end of document / end of text
# tokenizer.encode('<|endoftext|>', allowed_special={'<|endoftext|>'})
# then you get [50256] which means "end of text"

Vocab Size:: 50,257
Special tokens: {'<|endoftext|>'}


In [4]:
class StoryDataset:
    
    def __init__(self, stories, maxlen, tokenizer, add_endoftext=True):
        self.stories = stories
        self.maxlen = maxlen
        self.tokenizer = tokenizer
        self.add_endoftext = add_endoftext
         

    def __len__(self):
        return len(self.stories)

    def __getitem__(self, idx): # ith story
        story = self.stories[idx]
        if self.add_endoftext and not story.endswith('<|endoftext|>'):
            story = story + '<|endoftext|>'
        tokens = self.tokenizer.encode(story, allowed_special={'<|endoftext|>'})

        if len(tokens) > self.maxlen: # is story is too long cut it down to maxlen
            tokens = tokens[:self.maxlen]
            
        tokens.extend([0] * (self.maxlen - len(tokens))) # pad with 0s to maxlen
        return jnp.array(tokens, dtype=jnp.int32)

'''
e.g.
stories = [
    "Once upon a time, there was a cat.",
    "A little dog found a red ball.",
    "Mia went to the park with her dad."
]

stories[0] = "Once upon a time, there was a cat."
tokenizer.encode(stories[0], allowed_special={'<|endoftext|>'})


'''

'\ne.g.\nstories = [\n    "Once upon a time, there was a cat.",\n    "A little dog found a red ball.",\n    "Mia went to the park with her dad."\n]\n\nstories[0] = "Once upon a time, there was a cat."\ntokenizer.encode(stories[0], allowed_special={\'<|endoftext|>\'})\n\n\n'

In [5]:
shuffled_sampler = grain.IndexSampler(
    num_records = 10,
    shuffle=True,
    seed = 42,
    shard_options = grain.NoSharding(),
    num_epochs = 1
    )

def print_sampler_example(sampler, name):
    print(f"\n{name}")
    for i, idx in enumerate(sampler):
        print(f"Record {i}: {idx}")

print_sampler_example(shuffled_sampler, "Shuffled sampler")

# print 10 samples
    


Shuffled sampler
Record 0: RecordMetadata(index=0, record_key=8, rng=Generator(Philox))
Record 1: RecordMetadata(index=1, record_key=6, rng=Generator(Philox))
Record 2: RecordMetadata(index=2, record_key=7, rng=Generator(Philox))
Record 3: RecordMetadata(index=3, record_key=9, rng=Generator(Philox))
Record 4: RecordMetadata(index=4, record_key=0, rng=Generator(Philox))
Record 5: RecordMetadata(index=5, record_key=5, rng=Generator(Philox))
Record 6: RecordMetadata(index=6, record_key=1, rng=Generator(Philox))
Record 7: RecordMetadata(index=7, record_key=2, rng=Generator(Philox))
Record 8: RecordMetadata(index=8, record_key=4, rng=Generator(Philox))
Record 9: RecordMetadata(index=9, record_key=3, rng=Generator(Philox))


In [6]:
batch_op_keep = grain.Batch(
    batch_size=32,
    drop_remainder=False
)

In [7]:
def create_dataloader(
    stories,
    tokenizer,
    maxlen,
    batch_size,
    shuffle = False,
    num_epochs = 1,
    seed = 42,
    worker_count = 0
):
    dataset = StoryDataset(stories, maxlen, tokenizer) #my dataset object
    estimated_batches = (len(dataset) // batch_size) * num_epochs

    sampler = grain.IndexSampler(
        num_records=len(dataset), # 1000
        shuffle=shuffle,
        seed=seed,
        shard_options=grain.NoSharding(),
        num_epochs=num_epochs
    )
    dataloader = grain.DataLoader(
        data_source=dataset,
        sampler=sampler,
        operations=[
            grain.Batch(batch_size=batch_size, drop_remainder=True)
        ],
        worker_count=worker_count
    )
    
    return dataloader, estimated_batches

'''
def create_dataloader():
    dataset = storydataset()
    estimatedbatches = len(dataset) // batch_size
    sampler = grain.Dataloader()
    
    return dataloader, estimated batches
    
    
'''

'\ndef create_dataloader():\n    dataset = storydataset()\n    estimatedbatches = len(dataset) // batch_size\n    sampler = grain.Dataloader()\n\n    return dataloader, estimated batches\n\n\n'

In [8]:
stories = load_stories_from_file(
    "validation.csv", 
    max_stories=100
)

In [9]:
stories[0]

'Spot. Spot saw the shiny car and said, "Wow, Kitty, your car is so bright and clean!" Kitty smiled and replied, "Thank you, Spot. I polish it every day."\n\nAfter playing with the car, Kitty and Spot felt thirsty. They found a small pond with clear water. They drank the water and felt very happy. They played together all day and became best friends.'

In [10]:
dataloader, batches_per_epoch = create_dataloader(
    stories=stories,
    tokenizer=tokenizer,
    maxlen=128,
    batch_size=32,
    shuffle=False,
    num_epochs=1,
    seed=42,
    worker_count=0  # Single process for experimentation
)

print(f"\nDataLoader created successfully:")
print(f"Will produce {batches_per_epoch} batches per epoch")


DataLoader created successfully:
Will produce 3 batches per epoch


In [ ]:
next(iter(dataloader)) 

array([[32565,    13, 15899, ...,     0,     0,     0],
       [ 7454,  2402,   257, ...,   371, 23536,   373],
       [ 7454,  2402,   257, ...,   257,  1263,  1545],
       ...,
       [ 7454,  2402,   257, ...,   673,  1422,   470],
       [ 3198,  1110,    11, ...,  3424,    11,  2266],
       [ 7454,  2402,   257, ...,   422,   326,  1110]],
      shape=(32, 128), dtype=int32)